# General information

Example colab for SigLIP 2 models described in the [SigLIP 2 paper](https://arxiv.org/abs/2502.14786).

**These models are not official Google products and were trained and released for research purposes.**

If you find these model(s) useful for your research, consider citing

```
@article{tschannen2025siglip,
  title={SigLIP 2: Multilingual Vision-Language Encoders with Improved Semantic Understanding, Localization, and Dense Features},
  author={Tschannen, Michael and Gritsenko, Alexey and Wang, Xiao and Naeem, Muhammad Ferjad and Alabdulmohsin, Ibrahim and Parthasarathy, Nikhil and Evans, Talfan and Beyer, Lucas and Xia, Ye and Mustafa, Basil and H\'enaff, Olivier and Harmsen, Jeremiah and Steiner, Andreas and Zhai, Xiaohua},
  year={2025},
  journal={arXiv preprint arXiv:2502.14786}
}
```

If you use our released models in your products, we appreciate any direct feedback. Please contact tschannen@google.com to get in touch.

This Colab was forked from the previous version
[SigLIP_demo.ipynb](https://colab.research.google.com/github/google-research/big_vision/blob/main/big_vision/configs/proj/image_text/SigLIP_demo.ipynb)

In [2]:
#@markdown # Environment setup
#@markdown Tested on GPU, TPU, and CPU with ViT-B/16 @256px.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Hide GPU from TensorFlow
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"


import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  # Explicitly disable TF GPU usage

import os
if 'NVIDIA_PRODUCT_NAME' in os.environ:
  !nvidia-smi
import jax
jax.devices()


%cd big_vision


import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import jax
import jax.numpy as jnp
import ml_collections

/home/valenbonas/Documents/Investigacion_doctorado/new_siglip2/big_vision


In [ ]:
#@markdown Image credit:

#@markdown - The apple and apple + iPod images are from OpenAI.
#@markdown - Cows on beach were created by Chitwan Saharia using the Imagen model and shared with permission.
#@markdown - [Cold drink on hot day](https://unsplash.com/fr/photos/hQHm2D1fH70).
#@markdown - [Hot drink on cold day](https://www.rawpixel.com/image/3282934).
#@markdown - The remaining pictures are personal photos taken by the SigLIP1 authors.

# !wget -q https://cdn.openai.com/multimodal-neurons/assets/apple/apple-ipod.jpg
# !wget -q https://cdn.openai.com/multimodal-neurons/assets/apple/apple-blank.jpg
# !wget -q 'https://images.unsplash.com/photo-1566467021888-b03548769dd1?ixlib=rb-4.0.3&q=85&fm=jpg&crop=entropy&cs=srgb&dl=svetlana-gumerova-hQHm2D1fH70-unsplash.jpg&w=640' -O cold_drink.jpg
# !wget -q 'https://images.rawpixel.com/image_1300/czNmcy1wcml2YXRlL3Jhd3BpeGVsX2ltYWdlcy93ZWJzaXRlX2NvbnRlbnQvbHIvdXB3azU4ODU5NzY1LXdpa2ltZWRpYS1pbWFnZS1rb3diMmhkeC5qcGc.jpg' -O hot_drink.jpg
# !wget -q https://storage.googleapis.com/big_vision/siglip/authors.jpg
# !wget -q https://storage.googleapis.com/big_vision/siglip/siglip.jpg
# !wget -q https://storage.googleapis.com/big_vision/siglip/caffeine.jpg
# !wget -q https://storage.googleapis.com/big_vision/siglip/robosign.jpg
# !wget -q https://storage.googleapis.com/big_vision/siglip/fried_fish.jpeg
# !wget -q 'https://pbs.twimg.com/media/FTyEyxyXsAAyKPc?format=jpg&name=small' -O cow_beach.jpg
# !wget -q 'https://storage.googleapis.com/big_vision/siglip/cow_beach2.jpg' -O cow_beach2.jpg
# !wget -q 'https://pbs.twimg.com/media/Frb6NIEXwAA8-fI?format=jpg&name=medium' -O mountain_view.jpg
# !wget -q https://storage.googleapis.com/big_vision/siglip2/siglip2_screenshot.jpg


/home/valenbonas/miniconda3/envs/siglip2/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


# Choose and load model, perform inference

In [ ]:
# Pick your hero: (WHEN CHANGING THIS, RERUN IMAGE/TEXT EMBEDDING CELLS)
# Give this cell 1-3mins.

# VARIANT, RES = 'B/32', 256
# VARIANT, RES = 'B/16', 224
# VARIANT, RES = 'B/16', 256
# VARIANT, RES = 'B/16', 384
# VARIANT, RES = 'B/16', 512
# VARIANT, RES, N_PATCHES = 'B/16', 'naflex', 256
# VARIANT, RES = 'L/16', 256
# VARIANT, RES = 'L/16', 384
# VARIANT, RES = 'L/16', 512
# VARIANT, RES = 'So400m/16', 256
# VARIANT, RES = 'So400m/16', 384
# VARIANT, RES = 'So400m/16', 512
VARIANT, RES, N_PATCHES = 'So400m/16', 'naflex', 256
# VARIANT, RES = 'So400m/14', 224
# VARIANT, RES = 'So400m/14', 384
# VARIANT, RES = 'g-opt/16', 256
# VARIANT, RES = 'g-opt/16', 384

CKPT = f'siglip2_{VARIANT.lower().replace("/", "")}_{RES}.npz'
TXTVARIANT, PATCH_SIZE = VARIANT.split('/')
EMBDIM = {'B': 768, 'L': 1024, 'So400m': 1152, 'g-opt': 1536}[TXTVARIANT]
# Note: The g-opt vision encoder is paired with a So400m text encoder
TXTVARIANT = 'So400m' if TXTVARIANT == 'g-opt' else TXTVARIANT
PATCH_SIZE = int(PATCH_SIZE)
VOCAB = 256_000
SEQLEN = 64

# It is significantly faster to first copy the checkpoint (30s vs 8m30 for B and 1m vs ??? for L)
!test -f /tmp/{CKPT} || gsutil cp gs://big_vision/siglip2/{CKPT} /tmp/

import big_vision.models.proj.image_text.two_towers as model_mod

model_cfg = ml_collections.ConfigDict(dict(
    image_model='vit',
    image=dict(
        pool_type='map',
        scan=True,
        variant=VARIANT,
    ),
    text_model='proj.image_text.text_transformer',
    text=dict(
        scan=True,
        variant=TXTVARIANT,
        vocab_size=256_000,
    ),
    out_dim=[None, EMBDIM],
    bias_init=-10,  # without this arg, no "b" param is added
))
if RES == 'naflex':
  model_cfg.image_model = 'proj.image_text.naflex_vit'
  model_cfg.image = dict(
      pool_type='map',
      nposemb=16,
      posemb='learn_2d',
      variant='B'
  )

model = model_mod.Model(**model_cfg)

# Using `init_params` is slower but will lead to `load` below performing sanity-checks.
# init_params = jax.jit(model.init, backend="cpu")(jax.random.PRNGKey(42), jnp.zeros([1, RES, RES, 3], jnp.float32), jnp.zeros([1, SEQLEN], jnp.int32))['params']
init_params = None  # Faster but bypasses loading sanity-checks.

params = model_mod.load(init_params, f'/tmp/{CKPT}', model_cfg)

In [5]:
#@title Load and embed images

import big_vision.pp.builder as pp_builder
import big_vision.pp.ops_general
import big_vision.pp.ops_image
import big_vision.pp.ops_text
import big_vision.pp.proj.image_text.ops_naflex
import big_vision.pp.proj.paligemma.ops
import PIL

images = [PIL.Image.open(fname) for fname in [
    'apple-ipod.jpg',
    'apple-blank.jpg',
    'cold_drink.jpg',
    'hot_drink.jpg',
    'caffeine.jpg',
    'siglip.jpg',
    'authors.jpg',
    'robosign.jpg',
    'cow_beach.jpg',
    'cow_beach2.jpg',
    'mountain_view.jpg',
]]

get_pp_naflex = lambda n_patches, patch_size: pp_builder.get_preprocess_fn('|'.join((
    'value_range(-1, 1)',
    f'resize_to_sequence({patch_size}, {n_patches}, outkey="image")',
    f'patchify({patch_size}, key="image")',
    'flatten(["image"])',
    f'pad_to_shape(key="image/patches", shape=({n_patches}, None))',
    f'pad_to_shape(key="image/type", shape=({n_patches},))',
    f'pad_to_shape(key="image/yidx", shape=({n_patches},))',
    f'pad_to_shape(key="image/xidx", shape=({n_patches},))',
    'tuplify(["image/patches", "image/type", "image/yidx", "image/xidx"], "image")',
)))

if RES == 'naflex':
  # See section "NaFlex" below for a comparison of this preprocessing with
  # resize to square.
  pp_img = get_pp_naflex(N_PATCHES, PATCH_SIZE)
  imgs = [pp_img(dict(image=np.array(image)))['image'] for image in images]
  imgs = tuple(np.stack(_) for _ in zip(*imgs))
  print('imgs', tuple(x.shape for x in imgs))
else:
  pp_img = pp_builder.get_preprocess_fn(f'resize({RES})|value_range(-1, 1)')
  imgs = np.array([pp_img({'image': np.array(image)})['image'] for image in images])
  print('imgs', imgs.shape)

zimg, _, out = model.apply({'params': params}, imgs, None)

print('zimg', zimg.shape)

imgs (11, 384, 384, 3)
zimg (11, 1024)
